<a href="https://colab.research.google.com/github/Innovatewithapple/LangChain-Basic/blob/main/PaperRag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-huggingface langchain_community pymupdf

In [ ]:
#Documents Loader
from langchain_community.document_loaders import PyMuPDFLoader
import pymupdf
import re
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')
from langchain_core.documents import Document
import torch
from sentence_transformers import SentenceTransformer,CrossEncoder
from transformers import AutoTokenizer,AutoModelForCausalLM

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('huggingfaceToken'))

In [ ]:
#----------Load Encoder Model------------#
encoder_Tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5")
encoder_model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5',trust_remote_code=True)

#---------Load Decoder Models-----------#
decoder_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", trust_remote_code=True)
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token  # ← add this
llm = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct",dtype=torch.float16,device_map="auto",trust_remote_code=True)

# ---- LOAD RERANKER (once at startup) ----
reranking_model = CrossEncoder('BAAI/bge-reranker-large',device='cuda:0')

In [ ]:
def load_Process(path):
  doc = pymupdf.open(path)
  pages = []
  for page_no,page in enumerate(doc):
    page = page.get_text()
    clean_text = re.sub(r'\n{3,}', '\n\n', page) #Removes excessive empty lines.
    clean_text = re.sub(r'(?<!\n)\n(?!\n)', ' ', clean_text) #This fixes broken sentences. next char or previous char shouldn't be in next line
    clean_text = re.sub(r' {2,}', ' ', clean_text) #Removes multiple spaces.
    pages.append({
        'page_no':page_no,
        'page':clean_text
    })

  return pages

files = load_Process('/content/NIPS-2017-attention-is-all-you-need-Paper.pdf')

In [ ]:
files[0]['page']

In [ ]:
def Create_parent_child_chunks(pages,child_token_size,child_overlap_size,parent_token_size,parent_overlap_size):
  for page in pages:
    parent_chunks = []
    parent_current_chunk=[]
    parent_current_len=0

    sentence = sent_tokenize(text=page['page'])
    parent_token_len = [len(decoder_tokenizer.encode(sent,add_special_tokens=False)) for sent in sentence]

    for parent_sent,parent_token in zip(sentence,parent_token_len):
      if parent_current_len + parent_token < parent_token_size:
        parent_current_chunk.append(parent_sent)
        parent_current_len += parent_token
      else:
        if parent_current_chunk:
          parent_chunks.append(" ".join(parent_current_chunk))
        parent_current_chunk = parent_current_chunk[-parent_overlap_size:] + [parent_sent]
        parent_current_len = sum(parent_current_chunk[-parent_overlap_size:]) + parent_token

    if parent_current_chunk:
      parent_chunks.append(" ".join(parent_current_chunk))

    #-------Child Chunks-------!
    for p_idx,parent_chunk in enumerate(parent_chunks):
      parent_sentence = sent_tokenize(parent_chunk)
      parent_chunk_total_len = [len(encoder_Tokenizer.encode(parent_sentence,add_special_tokens=False)) for sent in parent_sentence]

      child_chunks = []
      child_current_chunk=[]
      child_current_len=0

      for parent_sent,parent_token in zip(parent_sentence,parent_chunk_total_len):
        if child_current_len + parent_token < child_token_size:
          child_current_chunk.append(parent_sent)
          child_current_len += parent_token
        else:
          if child_current_chunk:
            child_chunks.append(" ".join(child_current_chunk))
          child_current_chunk = child_current_chunk[-child_overlap_size:] + [parent_sent]
          child_current_len = sum(len(child_current_chunk)) + parent_token

      if child_current_chunk:
        child_chunks.append(" ".join(child_current_chunk))

      #-----Result----!
      results = []
      for child_idx,child in enumerate(child_chunks):
          results.append(
              Document(
                  page_content=child,
                  metadata={
                  'page_no':page['page_no'],
                  'parent_text':parent_chunk,
                  'parent_idx':p_idx,
                  'child_id':f'{p_idx}_{child_idx}'
                  }
                  ))
  return results

In [ ]:
all_chunks = Create_parent_child_chunks(pages=files,child_token_size=300,child_overlap_size=3,parent_token_size=600,parent_overlap_size=5)